### Neural Network from scratch

In [21]:
import numpy as np
from sklearn import datasets

# 1. Initialize hyperparameters
input_dim, h_dim, out_dim = 4, 10, 3  # Increased h_dim for better capacity
batch_size = 16
num_epoch = 100
alpha = 0.05

# 2. Load and prepare dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Neural networks converge much faster when input features are on the same scale.
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

def to_one_hot(y, out_dim):
    return np.eye(out_dim)[y]

Y_oh = to_one_hot(y, out_dim)

# Initial shuffle and split
indices = np.arange(len(X))
np.random.shuffle(indices)
X, Y_oh = X[indices], Y_oh[indices]

X_train, y_train = X[:120], Y_oh[:120]
X_test, y_test = X[120:], Y_oh[120:]

# 3. Initialize weights and biases
# Use He Initialization: np.sqrt(2/n) helps prevent vanishing/exploding gradients.
W1 = np.random.randn(input_dim, h_dim) * np.sqrt(2/input_dim)
B1 = np.zeros((1, h_dim))
W2 = np.random.randn(h_dim, out_dim) * np.sqrt(2/h_dim)
B2 = np.zeros((1, out_dim))

def ReLU(X):
    return np.maximum(X, 0)

def softmax(X):
    # axis=1 is CRITICAL for mini-batches to normalize across classes, not samples.
    exps = np.exp(X - np.max(X, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

def loss(Y, Z):
    # Cross-Entropy loss: we add 1e-15 to avoid log(0) which results in NaN.
    return np.mean(-np.sum(Y * np.log(Z + 1e-15), axis=1))

def feedforward(W1, W2, B1, B2, X):
    A1 = X @ W1 + B1
    H1 = ReLU(A1)
    A2 = H1 @ W2 + B2
    Z = softmax(A2)
    return A1, H1, A2, Z

def Backward(A1, H1, A2, Z, Y, X):
    m = X.shape[0] # Use dynamic batch size (last batch might be smaller)
    
    dE_dA2 = Z - Y         
    dE_dW2 = (H1.T @ dE_dA2) / m  # CRITICAL: Average gradient by dividing by batch size m
    dE_dB2 = np.sum(dE_dA2, axis=0, keepdims=True) / m      
    
    dE_dH1 = dE_dA2 @ W2.T   
    dE_dA1 = dE_dH1 * (A1 > 0) # ReLU derivative
    
    dE_dW1 = (X.T @ dE_dA1) / m
    dE_dB1 = np.sum(dE_dA1, axis=0, keepdims=True) / m       
    return dE_dW2, dE_dW1, dE_dB2, dE_dB1

def Update(dW2, dW1, dB2, dB1, W1, W2, B1, B2):
    W1 -= alpha * dW1
    W2 -= alpha * dW2
    B1 -= alpha * dB1
    B2 -= alpha * dB2
    return W1, W2, B1, B2

def predict(X):
    _, _, _, Z = feedforward(W1, W2, B1, B2, X)
    return np.argmax(Z, axis=1)

def accuracy(x_data, y_data):
    prediction = predict(x_data)
    y_true = np.argmax(y_data, axis=1)
    return np.mean(prediction == y_true)

# 4. Training Loop
for epoch in range(num_epoch):
    # Shuffle training data every epoch to improve generalization
    indices = np.arange(len(X_train))
    np.random.shuffle(indices)
    X_train, y_train = X_train[indices], y_train[indices]
    
    epoch_losses = [] # Reset loss list every epoch
      
    for i in range(0, len(X_train), batch_size):
        X_sample = X_train[i:i+batch_size]
        Y_sample = y_train[i:i+batch_size]
        
        # Forward -> Backward -> Update
        A1, H1, A2, Z = feedforward(W1, W2, B1, B2, X_sample)
        E = loss(Y_sample, Z)
        dW2, dW1, dB2, dB1 = Backward(A1, H1, A2, Z, Y_sample, X_sample)
        W1, W2, B1, B2 = Update(dW2, dW1, dB2, dB1, W1, W2, B1, B2)
        
        epoch_losses.append(E)

    if (epoch + 1) % 10 == 0:
        acc_tr = accuracy(X_train, y_train)
        acc_ts = accuracy(X_test, y_test)
        print(f"Epoch: {epoch+1:3} | Loss: {np.mean(epoch_losses):.4f} | Acc Train: {acc_tr:.4f} | Acc Test: {acc_ts:.4f}")

Epoch:  10 | Loss: 0.3986 | Acc Train: 0.8500 | Acc Test: 0.8333
Epoch:  20 | Loss: 0.2817 | Acc Train: 0.9000 | Acc Test: 0.9000
Epoch:  30 | Loss: 0.2335 | Acc Train: 0.9333 | Acc Test: 0.9333
Epoch:  40 | Loss: 0.1897 | Acc Train: 0.9500 | Acc Test: 0.9333
Epoch:  50 | Loss: 0.1516 | Acc Train: 0.9417 | Acc Test: 0.9333
Epoch:  60 | Loss: 0.1307 | Acc Train: 0.9583 | Acc Test: 0.9333
Epoch:  70 | Loss: 0.1171 | Acc Train: 0.9583 | Acc Test: 0.9333
Epoch:  80 | Loss: 0.1094 | Acc Train: 0.9583 | Acc Test: 0.9333
Epoch:  90 | Loss: 0.0967 | Acc Train: 0.9667 | Acc Test: 0.9333
Epoch: 100 | Loss: 0.0888 | Acc Train: 0.9667 | Acc Test: 0.9333


### Neural Network with Pytorch

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Loading Data 
iris = datasets.load_iris()
X = iris.data
y = iris.target

# split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Standardizing data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)  # only transforem

# Convert Numpy arrays to Pytorch Tensor
# X is float32, y is int64 (LongTensor)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.int64)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.int64)

# Create Dataloader (Handle batches and shuffling automatically)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)

# 2. Define the Neural Network Architecture
class IrisNet(nn.Module):
    def __init__(self, input_dim, h_dim, out_dim):
        super(IrisNet, self).__init__()
        # nn.Linear automatically initializes weights and biases
        self.fc1 = nn.Linear(input_dim, h_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(h_dim, out_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        # Note: No Softmax here!
        # nn.CrossEntropyloss in Pytorch combines Softmax and Loss in one function
        return x

# 3 Initialize Model, Loss, and Optimizer
model = IrisNet(input_dim=4, h_dim=5, out_dim=3)
criterion = nn.CrossEntropyLoss()  # loss fnction
optimizer = optim.SGD(model.parameters(), lr=0.05)

# 4. Training Looop
num_epochs=100

for epoch in range(num_epochs):
    model.train() 
    epoch_losses = 0

    for X_batch, y_batch in train_loader:
        # Forward Pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward Pass 
        optimizer.zero_grad() # Clear previous gradients
        loss.backward()       # Automatically calculate all derivatives (dW, dB)

        # Update Weights
        optimizer.step()      # W = W - lr * dW 

        epoch_losses += loss.item()

    if (epoch+1) % 10 == 0:
        model.eval() # Set model to evaluation mode
        with torch.no_grad():
            train_preds = model(X_train_tensor)  # count accuracy on train
            train_acc = (torch.argmax(train_preds, dim=1)==y_train_tensor).float().mean() 

            test_preds = model(X_test_tensor)# count accuracy on test
            test_acc = (torch.argmax(test_preds, dim=1)==y_test_tensor).float().mean()

        print(f"Epochs [{epoch+1:3}/{num_epochs:3}] | Loss: {epoch_losses:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
            

Epochs [ 10/100] | Loss: 4.4158 | Train Acc: 0.8833 | Test Acc: 0.9000
Epochs [ 20/100] | Loss: 2.6744 | Train Acc: 0.9167 | Test Acc: 1.0000
Epochs [ 30/100] | Loss: 2.0401 | Train Acc: 0.9333 | Test Acc: 1.0000
Epochs [ 40/100] | Loss: 1.7706 | Train Acc: 0.9500 | Test Acc: 1.0000
Epochs [ 50/100] | Loss: 1.4400 | Train Acc: 0.9583 | Test Acc: 1.0000
Epochs [ 60/100] | Loss: 1.2884 | Train Acc: 0.9583 | Test Acc: 1.0000
Epochs [ 70/100] | Loss: 1.1108 | Train Acc: 0.9667 | Test Acc: 1.0000
Epochs [ 80/100] | Loss: 1.0250 | Train Acc: 0.9667 | Test Acc: 1.0000
Epochs [ 90/100] | Loss: 0.9200 | Train Acc: 0.9667 | Test Acc: 1.0000
Epochs [100/100] | Loss: 0.8890 | Train Acc: 0.9667 | Test Acc: 1.0000
